In [ ]:
!sudo apt-get update
!sudo apt-get install stockfish -y
%pip install flask-cors pyngrok flask

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]  
Get:13 https://ppa.laun

In [1]:
!pkill -f ngrok
!pkill -f flask

In [ ]:
# 1. 環境依賴安裝 (確保 Colab 擁有所有必要的擴充套件)
%pip install flask-cors pyngrok
# !apt-get install stockfish  # 若虛擬機尚未安裝引擎，請取消註解並執行此行

# 2. 核心程式碼
import subprocess
import shutil
import time
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

stockfish_path = shutil.which('stockfish') or '/usr/games/stockfish'

print("⚙️ 正在召喚雲端運算核心：分配 4GB 記憶體並啟動 NNUE...")
engine = subprocess.Popen(
    stockfish_path,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    universal_newlines=True,
    bufsize=1
)

engine.stdin.write("uci\n")
engine.stdin.write("setoption name Threads value 2\n")
engine.stdin.write("setoption name Hash value 4096\n")
engine.stdin.write("setoption name Use NNUE value true\n")
engine.stdin.write("isready\n")
engine.stdin.flush()

while True:
    line = engine.stdout.readline()
    if 'readyok' in line:
        print("✅ 運算核心已就緒")
        break

app = Flask(__name__)
CORS(app)
engine_lock = threading.Lock()

@app.route('/analyze', methods=['GET'])
def analyze():
    fen = request.args.get('fen')
    multipv = request.args.get('multipv', '3')
    depth = 22 # 深度 22 
    
    start_time = time.time()
    try:
        with engine_lock:
            engine.stdin.write("stop\n")
            engine.stdin.write("isready\n")
            engine.stdin.flush()
            while True:
                line = engine.stdout.readline()
                if 'readyok' in line: break
                    
            engine.stdin.write(f"setoption name MultiPV value {multipv}\n")
            engine.stdin.write(f"position fen {fen}\n")
            engine.stdin.write(f"go depth {depth}\n")
            engine.stdin.flush()
            
            analysis_lines = []
            while True:
                line = engine.stdout.readline()
                if 'info depth' in line and 'multipv 1' in line:
                    d = line.split('depth ')[1].split(' ')[0]
                    print(f"🧠 思考中... 深度: {d} | 模式: {multipv} 種著法", end='\r')
                if f'info depth {depth}' in line and 'multipv' in line:
                    analysis_lines.append(line.strip())
                if 'bestmove' in line:
                    calc_time = time.time() - start_time
                    print(f"\n🎯 運算完成！耗時: {calc_time:.2f} 秒")
                    break
        return jsonify({"analysis": analysis_lines[-int(multipv):]})
    except Exception as e:
        print(f"\n❌ 錯誤: {e}")
        return jsonify({"error": str(e)}), 500

try:
    # ⚠️ 這裡要換成你自己的 ngrok Token！
    ngrok.set_auth_token("3CUqUEFiCwKn4BA9rHGdXBoAMoV_3iamaRvPerV94MbKoHHnA")
    public_url = ngrok.connect(5000, domain="tuesday-spoilage-ceremony.ngrok-free.dev")
    print(f"\n✅ 雲端大門已開啟！")
    print(f"🔗 專屬網址: {public_url.public_url}")
except Exception as e:
    print(f"❌ ngrok 連線失敗: {e}")

app.run(port=5000)

⚙️ 正在召喚雲端運算核心：分配 4GB 記憶體並啟動 NNUE...
✅ 運算核心已就緒

✅ 雲端大門已開啟！
🔗 專屬網址: https://tuesday-spoilage-ceremony.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 17:58:19] "OPTIONS /analyze?fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR%20b%20KQkq%20e3%200%201&multipv=3 HTTP/1.1" 200 -


INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 17:58:47] "GET /analyze?fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR%20b%20KQkq%20e3%200%201&multipv=3 HTTP/1.1" 200 -


🧠 思考中... 深度: 22 | 模式: 3 種著法
🎯 運算完成！耗時: 27.90 秒


INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 17:58:52] "OPTIONS /analyze?fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR%20b%20KQkq%20e3%200%201&multipv=1 HTTP/1.1" 200 -


...

INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:32:24] "OPTIONS /analyze?fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR%20b%20KQkq%20e3%200%201&multipv=3 HTTP/1.1" 200 -


INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:32:26] "GET /analyze?fen=rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR%20w%20KQkq%20-%200%201&multipv=3 HTTP/1.1" 200 -


🧠 思考中... 深度: 22 | 模式: 3 種著法
🎯 運算完成！耗時: 3.31 秒


INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 18:32:33] "GET /analyze?fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR%20b%20KQkq%20e3%200%201&multipv=3 HTTP/1.1" 200 -


🧠 思考中... 深度: 22 | 模式: 3 種著法
🎯 運算完成！耗時: 8.74 秒
